<a href="https://colab.research.google.com/github/alishbakhan21/pfdhfr-docking-resistance/blob/main/Copy_of_01_setup_and_structures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Question: Can free docking tools reproduce a known drug pose in P. falciparum DHFR-TS, and do they show differences between the drug-sensitive (wild type) and drug-resistant (quadruple mutant) enzyme?

In [ ]:
!pip install -q rdkit meeko gemmi
!apt-get -qq install -y openbabel > /dev/null
!wget -q https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64 -O vina
!chmod +x vina
!./vina --version

AutoDock Vina v1.2.5


In [ ]:
!wget -q https://files.rcsb.org/download/1J3I.pdb
!wget -q https://files.rcsb.org/download/1J3K.pdb
!ls -lh *.pdb

-rw-r--r-- 1 root root 879K Sep 22  2025 1J3I.pdb
-rw-r--r-- 1 root root 151K Nov  5 08:42 1J3I_prot.pdb
-rw-r--r-- 1 root root 2.2K Nov  5 08:42 1J3I_ref.pdb
-rw-r--r-- 1 root root 912K Sep 22  2025 1J3K.pdb
-rw-r--r-- 1 root root 150K Nov  5 08:42 1J3K_prot.pdb
-rw-r--r-- 1 root root 2.2K Nov  5 08:42 1J3K_ref.pdb
-rw-r--r-- 1 root root 5.2K Nov  5 09:21 redock_mut_pose1.pdb
-rw-r--r-- 1 root root 5.2K Nov  5 09:07 redock_wt_pose1.pdb


In [ ]:
import gemmi

for pdb in ["1J3I.pdb", "1J3K.pdb"]:
    st = gemmi.read_structure(pdb)
    print(pdb)
    for ch in st[0]:
        ligs = sorted({r.name for r in ch if r.het_flag == "H" and r.name != "HOH"})
        print("  chain", ch.name, "|", len(ch), "residues | ligands:", ligs)

1J3I.pdb
  chain A | 381 residues | ligands: ['NDP', 'WRA']
  chain B | 279 residues | ligands: ['NDP', 'WRA']
  chain C | 617 residues | ligands: ['UMP']
  chain D | 621 residues | ligands: ['UMP']
1J3K.pdb
  chain A | 461 residues | ligands: ['NDP', 'WRA']
  chain B | 325 residues | ligands: ['NDP', 'WRA']
  chain C | 755 residues | ligands: ['UMP']
  chain D | 806 residues | ligands: ['UMP']


Both structures contain the DHFR domain with WR99210 (WRA) and NADPH (NDP) bound in chains A and B, and the TS domain with dUMP in chains C and D. Residue counts include water molecules. For docking I'll use chain A only and keep NADPH, because the drug binds next to it. I'll remove water, dUMP and the other chains, and save the crystal pose of WR99210 to check my docking against.

In [ ]:
import gemmi

for pdb in ["1J3I", "1J3K"]:
    st = gemmi.read_structure(pdb + ".pdb")
    st.setup_entities()
    out = gemmi.Structure()
    out.add_model(gemmi.Model("1"))
    ch = gemmi.Chain("A")
    for res in st[0]["A"]:
        if res.name == "WRA":
            ch.add_residue(res)
    out[0].add_chain(ch)
    out.write_pdb(pdb + "_ref.pdb")
    print(pdb, "reference residues:", len(ch))

1J3I reference residues: 1
1J3K reference residues: 1


I extracted WR99210 from each structure and saved it separately. This is the crystal pose the known answer I'll later check my docking against.

In [ ]:
import gemmi

for pdb in ["1J3I", "1J3K"]:
    st = gemmi.read_structure(pdb + ".pdb")
    st.setup_entities()
    st.remove_alternative_conformations()
    st.remove_hydrogens()
    st.remove_waters()
    out = gemmi.Structure()
    out.add_model(gemmi.Model("1"))
    ch = gemmi.Chain("A")
    n = 0
    for res in st[0]["A"]:
        if res.name != "WRA":
            ch.add_residue(res)
            n += 1
    out[0].add_chain(ch)
    out.write_pdb(pdb + "_prot.pdb")
    print(pdb, "kept", n, "residues in chain A")

1J3I kept 224 residues in chain A
1J3K kept 222 residues in chain A


Built the receptor files: kept chain A and NADPH, removed water, the drug and the other chains. Removing the drug is essential, since docking has to find the pocket without being shown the answer. Kept 224 residues for 1J3I and 222 for 1J3K.

In [ ]:
import gemmi

for pdb in ["1J3I", "1J3K"]:
    st = gemmi.read_structure(pdb + "_prot.pdb")
    ligs = sorted({r.name for r in st[0]["A"] if r.het_flag == "H"})
    print(pdb, "| ligands present:", ligs)

1J3I | ligands present: ['NDP']
1J3K | ligands present: ['NDP']


Checked what remained. Both files contain only NADPH, confirming the drug and water were removed and the cofactor was kept.

In [ ]:
!obabel 1J3I_prot.pdb -O 1J3I_prot.pdbqt -xr --partialcharge gasteiger
!obabel 1J3K_prot.pdb -O 1J3K_prot.pdbqt -xr --partialcharge gasteiger
!ls -lh *_prot.pdbqt

*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is 1J3I_prot.pdb)

1 molecule converted
*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is 1J3K_prot.pdb)

1 molecule converted
-rw-r--r-- 1 root root 149K Nov  5 09:59 1J3I_prot.pdbqt
-rw-r--r-- 1 root root 148K Nov  5 09:59 1J3K_prot.pdbqt


Converted the receptors to PDBQT, the format Vina reads, with Gasteiger charges and a rigid receptor. Open Babel warned that it could not kekulize aromatic bonds; this is common for protein PDB files and does not affect a rigid receptor, where bond orders aren't used.

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem

smiles = {
 "wr99210": "CC1(C)N=C(N)N=C(N)N1OCCCOc1cc(Cl)c(Cl)cc1Cl",
 "pyrimethamine": "CCc1nc(N)nc(N)c1-c1ccc(Cl)cc1",
 "cycloguanil": "CC1(C)N=C(N)N=C(N)N1c1ccc(Cl)cc1",
}

for name, smi in smiles.items():
    m = Chem.MolFromSmiles(smi)
    if m is None:
        print(name, "FAILED to parse")
        continue
    m = Chem.AddHs(m)
    AllChem.EmbedMolecule(m, randomSeed=42)
    AllChem.MMFFOptimizeMolecule(m)
    Chem.MolToMolFile(m, name + ".mol")
    print(name, "ok |", m.GetNumAtoms(), "atoms |", Chem.MolToSmiles(Chem.RemoveHs(m)))

wr99210 ok | 42 atoms | CC1(C)N=C(N)N=C(N)N1OCCCOc1cc(Cl)c(Cl)cc1Cl
pyrimethamine ok | 30 atoms | CCc1nc(N)nc(N)c1-c1ccc(Cl)cc1
cycloguanil ok | 31 atoms | CC1(C)N=C(N)N=C(N)N1c1ccc(Cl)cc1


I checked each SMILES against the RCSB Chemical Component Dictionary: WRA (WR99210, C₁₄H₁₈Cl₃N₅O₂), CP6 (pyrimethamine, C₁₂H₁₃ClN₄) and 1CY (cycloguanil, C₁₁H₁₄ClN₅). All three match. I used RCSB rather than PubChem because WRA is the ligand ID present in 1J3I and 1J3K, so this confirms my docking ligand is the same molecule as the crystal ligand. An earlier version of my WR99210 SMILES had two chlorines instead of three; this check caught it.

In [ ]:
for name in ["wr99210", "pyrimethamine", "cycloguanil"]:
      !obabel {name}.mol -O {name}.pdbqt --partialcharge gasteiger
      !ls -lh *.pdbqt

1 molecule converted
-rw-r--r-- 1 root root 149K Nov  5 09:59 1J3I_prot.pdbqt
-rw-r--r-- 1 root root 148K Nov  5 09:59 1J3K_prot.pdbqt
-rw-r--r-- 1 root root 2.1K Nov  5 08:55 cycloguanil.pdbqt
-rw-r--r-- 1 root root  21K Nov  5 09:11 dock_1J3I_cycloguanil.pdbqt
-rw-r--r-- 1 root root  22K Nov  5 09:11 dock_1J3I_pyrimethamine.pdbqt
-rw-r--r-- 1 root root  30K Nov  5 09:10 dock_1J3I_wr99210.pdbqt
-rw-r--r-- 1 root root  21K Nov  5 09:14 dock_1J3K_cycloguanil.pdbqt
-rw-r--r-- 1 root root  22K Nov  5 09:14 dock_1J3K_pyrimethamine.pdbqt
-rw-r--r-- 1 root root  30K Nov  5 09:13 dock_1J3K_wr99210.pdbqt
-rw-r--r-- 1 root root 2.2K Nov  5 08:55 pyrimethamine.pdbqt
-rw-r--r-- 1 root root  30K Nov  5 09:21 redock_mut.pdbqt
-rw-r--r-- 1 root root  30K Nov  5 09:06 redock_wt.pdbqt
-rw-r--r-- 1 root root  21K Nov  5 09:36 s1_1J3I_cycloguanil.pdbqt
-rw-r--r-- 1 root root  22K Nov  5 09:33 s1_1J3I_pyrimethamine.pdbqt
-rw-r--r-- 1 root root  30K Nov  5 09:24 s1_1J3I_wr99210.pdbqt
-rw-r--r-- 1 root roo

In [ ]:
import gemmi, numpy as np

st = gemmi.read_structure("1J3I_ref.pdb")
pos = np.array([[a.pos.x, a.pos.y, a.pos.z] for r in st[0]["A"] for a in r])
center = pos.mean(axis=0)
print("center:", np.round(center, 2))
print("box size: 22 x 22 x 22")

center: [28.02  6.08 59.79]
box size: 22 x 22 x 22


Docking box centred on the crystal ligand position, fixed 22 Å cube. Using the known ligand position is standard practice, but it means this is not a blind docking.

In [ ]:
import gemmi, numpy as np

st = gemmi.read_structure("1J3I_ref.pdb")
pos = np.array([[a.pos.x, a.pos.y, a.pos.z] for r in st[0]["A"] for a in r])
cx, cy, cz = pos.mean(axis=0)
print("center:", round(cx,2), round(cy,2), round(cz,2))

center: 28.02 6.08 59.79


In [ ]:
!./vina --receptor 1J3I_prot.pdbqt --ligand wr99210.pdbqt \
  --center_x {cx} --center_y {cy} --center_z {cz} \
    --size_x 22 --size_y 22 --size_z 22 \
      --exhaustiveness 16 --seed 42 \
        --out redock_wt.pdbqt

AutoDock Vina v1.2.5
#################################################################
# If you used AutoDock Vina in your work, please cite:          #
#                                                               #
# J. Eberhardt, D. Santos-Martins, A. F. Tillack, and S. Forli  #
# AutoDock Vina 1.2.0: New Docking Methods, Expanded Force      #
# Field, and Python Bindings, J. Chem. Inf. Model. (2021)       #
# DOI 10.1021/acs.jcim.1c00203                                  #
#                                                               #
# O. Trott, A. J. Olson,                                        #
# AutoDock Vina: improving the speed and accuracy of docking    #
# with a new scoring function, efficient optimization and       #
# multithreading, J. Comp. Chem. (2010)                         #
# DOI 10.1002/jcc.21334                                         #
#                                                               #
# Please see https://github.com/ccsb-scripps/AutoDock-V

Before comparing anything, I checked whether the method can reproduce a known answer. I removed WR99210 from the wild-type structure and asked Vina to find it again.

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign

!obabel redock_wt.pdbqt -O redock_wt_pose1.pdb -f 1 -l 1

tmpl = Chem.MolFromSmiles("CC1(C)N=C(N)N=C(N)N1OCCCOc1cc(Cl)c(Cl)cc1Cl")

ref  = Chem.MolFromPDBFile("1J3I_ref.pdb", removeHs=True)
dock = Chem.MolFromPDBFile("redock_wt_pose1.pdb", removeHs=True)
print("ref atoms:", ref.GetNumAtoms(), "| docked atoms:", dock.GetNumAtoms())

ref  = AllChem.AssignBondOrdersFromTemplate(tmpl, ref)
dock = AllChem.AssignBondOrdersFromTemplate(tmpl, dock)

rmsd = rdMolAlign.CalcRMS(dock, ref)
print("redocking RMSD = %.2f A" % rmsd)

1 molecule converted
ref atoms: 24 | docked atoms: 24
redocking RMSD = 1.95 A


[10:01:29] WARNING: More than one matching pattern found - picking one



Wild-type RMSD 1.95 Å. The usual success threshold is 2.0 Å, so this passes, but only just. I'm reporting the actual value rather than rounding it.
RDKit warned about multiple matching patterns because the molecule is symmetric; CalcRMS accounts for symmetry, so the value is correct. I used CalcRMS rather than GetBestRMS because GetBestRMS would move my pose onto the crystal pose first, hiding the positional error I'm measuring.

In [ ]:
import itertools
import re

def best_score(path):
    for line in open(path):
        if line.startswith("REMARK VINA RESULT"):
            return float(line.split()[3])
    return None

rows = []
for prot, lig in itertools.product(["1J3I", "1J3K"], ["wr99210", "pyrimethamine", "cycloguanil"]):
    out = f"dock_{prot}_{lig}.pdbqt"
    !./vina --receptor {prot}_prot.pdbqt --ligand {lig}.pdbqt \
      --center_x {cx} --center_y {cy} --center_z {cz} \
      --size_x 22 --size_y 22 --size_z 22 \
      --exhaustiveness 16 --seed 42 --out {out} > /dev/null 2>&1

    score = best_score(out)
    rows.append({"structure": prot, "ligand": lig, "score": score})
    print(prot, lig, score)

1J3I wr99210 -8.407
1J3I pyrimethamine -8.826
1J3I cycloguanil -9.315
1J3K wr99210 -8.626
1J3K pyrimethamine -7.851
1J3K cycloguanil -7.985


Scores in kcal/mol. Cycloguanil −9.32 (WT) vs −7.99 (mutant); pyrimethamine −8.83 vs −7.85; WR99210 −8.41 vs −8.63. The direction matches what is known clinically: the two antifolates lose activity against the quadruple mutant while WR99210 retains it.
Two limits. The size of the effect is wrong about 1 kcal/mol is roughly a five-fold affinity change, far less than the real loss of potency. And the ranking between ligands is wrong: Vina puts cycloguanil ahead of WR99210 on the wild type, which experiment does not support. So the method may track how one ligand changes between two structures, but it cannot rank different ligands against each other.

In [ ]:
import pandas as pd
df = pd.DataFrame(rows)
table = df.pivot(index="ligand", columns="structure", values="score")
table["difference"] = table["1J3K"] - table["1J3I"]
table

structure,1J3I,1J3K,difference
ligand,,,
cycloguanil,-9.315,-7.985,1.330
pyrimethamine,-8.826,-7.851,0.975
wr99210,-8.407,-8.626,-0.219


In [ ]:
st2 = gemmi.read_structure("1J3K_ref.pdb")
p2 = np.array([[a.pos.x,a.pos.y,a.pos.z] for r in st2[0]["A"] for a in r])
mx, my, mz = p2.mean(axis=0)

!./vina --receptor 1J3K_prot.pdbqt --ligand wr99210.pdbqt \
  --center_x {mx} --center_y {my} --center_z {mz} \
  --size_x 22 --size_y 22 --size_z 22 \
  --exhaustiveness 16 --seed 42 --out redock_mut.pdbqt

!obabel redock_mut.pdbqt -O redock_mut_pose1.pdb -f 1 -l 1
r2 = AllChem.AssignBondOrdersFromTemplate(tmpl, Chem.MolFromPDBFile("1J3K_ref.pdb", removeHs=True))
d2 = AllChem.AssignBondOrdersFromTemplate(tmpl, Chem.MolFromPDBFile("redock_mut_pose1.pdb", removeHs=True))
print("mutant redocking RMSD = %.2f A" % rdMolAlign.CalcRMS(d2, r2))

AutoDock Vina v1.2.5
#################################################################
# If you used AutoDock Vina in your work, please cite:          #
#                                                               #
# J. Eberhardt, D. Santos-Martins, A. F. Tillack, and S. Forli  #
# AutoDock Vina 1.2.0: New Docking Methods, Expanded Force      #
# Field, and Python Bindings, J. Chem. Inf. Model. (2021)       #
# DOI 10.1021/acs.jcim.1c00203                                  #
#                                                               #
# O. Trott, A. J. Olson,                                        #
# AutoDock Vina: improving the speed and accuracy of docking    #
# with a new scoring function, efficient optimization and       #
# multithreading, J. Comp. Chem. (2010)                         #
# DOI 10.1002/jcc.21334                                         #
#                                                               #
# Please see https://github.com/ccsb-scripps/AutoDock-V

[10:09:46] WARNING: More than one matching pattern found - picking one



Mutant RMSD 3.81 Å this fails. The same procedure that worked on the wild type did not reproduce the crystal pose here. That matters, because the resistance comparison depends on the mutant numbers. I didn't expect this and I'm keeping it in rather than reporting only the structure that worked.

In [ ]:
import statistics

for prot in ["1J3I", "1J3K"]:
    for lig in ["wr99210", "pyrimethamine", "cycloguanil"]:
        ss = []
        for seed in [1, 2, 3, 4, 5]:
            o = f"s{seed}_{prot}_{lig}.pdbqt"
            !./vina --receptor {prot}_prot.pdbqt --ligand {lig}.pdbqt \
              --center_x {cx} --center_y {cy} --center_z {cz} \
              --size_x 22 --size_y 22 --size_z 22 \
              --exhaustiveness 16 --seed {seed} --out {o} > /dev/null 2>&1
            ss.append(best_score(o))
        print(prot, lig, "mean %.2f  sd %.2f" % (statistics.mean(ss), statistics.stdev(ss)))

1J3I wr99210 mean -8.35  sd 0.06
1J3I pyrimethamine mean -8.82  sd 0.01
1J3I cycloguanil mean -9.32  sd 0.02
1J3K wr99210 mean -8.65  sd 0.03
1J3K pyrimethamine mean -7.85  sd 0.01
1J3K cycloguanil mean -7.98  sd 0.01


Standard deviations of 0.01–0.06 kcal/mol over five seeds, so the search converges reproducibly. This shows the result is repeatable, not that it is accurate — the scoring function's own error is much larger. It rules out random search noise and nothing more.

In [ ]:
!obabel redock_mut.pdbqt -O mut_pose.pdb -m

import glob, re
files = sorted(glob.glob("mut_pose*.pdb"), key=lambda s: int(re.findall(r"\d+", s)[-1]))
for f in files:
    m = Chem.MolFromPDBFile(f, removeHs=True)
    if m is None:
        print(f, "parse failed"); continue
    m = AllChem.AssignBondOrdersFromTemplate(tmpl, m)
    print(f, "RMSD %.2f A" % rdMolAlign.CalcRMS(m, r2))

9 molecules converted
9 files output. The first is mut_pose1.pdb
mut_pose1.pdb RMSD 3.81 A
mut_pose2.pdb RMSD 2.50 A
mut_pose3.pdb RMSD 4.18 A
mut_pose4.pdb RMSD 5.26 A
mut_pose5.pdb RMSD 7.99 A
mut_pose6.pdb RMSD 3.46 A
mut_pose7.pdb RMSD 4.06 A
mut_pose8.pdb RMSD 5.69 A
mut_pose9.pdb RMSD 3.62 A


Vina returns nine poses and I had only checked the top-ranked one, so I measured all nine. The best was pose 2 at 2.50 Å; the top-ranked pose was 3.81 Å and the rest ranged from 3.46 to 7.99 Å. No pose was within the 2.0 Å threshold, so the correct pose was not found at all this is a sampling failure rather than the scoring function misranking a correct pose. Pose 2 is in the right pocket but wrongly oriented.

In [ ]:
!./vina --receptor 1J3K_prot.pdbqt --ligand wr99210.pdbqt \
  --center_x {mx} --center_y {my} --center_z {mz} \
  --size_x 22 --size_y 22 --size_z 22 \
  --exhaustiveness 64 --seed 42 --out redock_mut_e64.pdbqt

!obabel redock_mut_e64.pdbqt -O mut64_pose1.pdb -f 1 -l 1
d3 = AllChem.AssignBondOrdersFromTemplate(tmpl, Chem.MolFromPDBFile("mut64_pose1.pdb", removeHs=True))
print("exhaustiveness 64 RMSD = %.2f A" % rdMolAlign.CalcRMS(d3, r2))

AutoDock Vina v1.2.5
#################################################################
# If you used AutoDock Vina in your work, please cite:          #
#                                                               #
# J. Eberhardt, D. Santos-Martins, A. F. Tillack, and S. Forli  #
# AutoDock Vina 1.2.0: New Docking Methods, Expanded Force      #
# Field, and Python Bindings, J. Chem. Inf. Model. (2021)       #
# DOI 10.1021/acs.jcim.1c00203                                  #
#                                                               #
# O. Trott, A. J. Olson,                                        #
# AutoDock Vina: improving the speed and accuracy of docking    #
# with a new scoring function, efficient optimization and       #
# multithreading, J. Comp. Chem. (2010)                         #
# DOI 10.1002/jcc.21334                                         #
#                                                               #
# Please see https://github.com/ccsb-scripps/AutoDock-V

Repeated the mutant redocking with exhaustiveness 64 instead of 16. The result was identical: same top pose, same score of −8.626, same RMSD of 3.81 Å. Increasing the search effort fourfold changed nothing, so this is not a sampling problem the search converges, but it converges on the wrong pose.

In [ ]:
!obabel redock_mut.pdbqt -O mut_pose.pdb -m

import glob, re
for f in sorted(glob.glob("mut_pose*.pdb"), key=lambda s: int(re.findall(r"\d+", s)[-1])):
    m = Chem.MolFromPDBFile(f, removeHs=True)
    if m is None:
        print(f, "parse failed")
        continue
    m = AllChem.AssignBondOrdersFromTemplate(tmpl, m)
    print(f, "RMSD %.2f A" % rdMolAlign.CalcRMS(m, r2))

9 molecules converted
9 files output. The first is mut_pose1.pdb
mut_pose1.pdb RMSD 3.81 A
mut_pose2.pdb RMSD 2.50 A
mut_pose3.pdb RMSD 4.18 A
mut_pose4.pdb RMSD 5.26 A
mut_pose5.pdb RMSD 7.99 A
mut_pose6.pdb RMSD 3.46 A
mut_pose7.pdb RMSD 4.06 A
mut_pose8.pdb RMSD 5.69 A
mut_pose9.pdb RMSD 3.62 A


Vina returns nine poses and I had only checked the top-ranked one, so I measured all nine. The best was pose 2 at 2.50 Å; the top-ranked pose was 3.81 Å and the rest ranged from 3.46 to 7.99 Å. No pose was within the 2.0 Å threshold, so the correct pose was not found at all a sampling failure rather than the scoring function misranking a correct pose. Pose 2 is in the right pocket but wrongly oriented.

In [ ]:
for f in ["1J3I_prot.pdbqt", "1J3K_prot.pdbqt"]:
    txt = open(f).read()
    print(f, "| NDP lines:", txt.count("NDP"), "| atom records:", txt.count("ATOM "))

1J3I_prot.pdbqt | NDP lines: 48 | atom records: 1895
1J3K_prot.pdbqt | NDP lines: 48 | atom records: 1885


I also checked whether receptor preparation differed between the two structures. Both PDBQT files contain NADPH (48 records each) and have nearly identical atom counts (1895 vs 1885), so the failure is not explained by a missing cofactor or a broken receptor file.

In [ ]:
!obabel redock_mut_e64.pdbqt -O m64.pdb -m

import glob, re
for f in sorted(glob.glob("m64*.pdb"), key=lambda s: int(re.findall(r"\d+", s)[-1])):
    m = Chem.MolFromPDBFile(f, removeHs=True)
    if m is None:
        print(f, "parse failed")
        continue
    m = AllChem.AssignBondOrdersFromTemplate(tmpl, m)
    print(f, "RMSD %.2f A" % rdMolAlign.CalcRMS(m, r2))

9 molecules converted
9 files output. The first is m641.pdb
m641.pdb RMSD 3.81 A
m642.pdb RMSD 2.45 A
m643.pdb RMSD 4.18 A
m644.pdb RMSD 6.80 A
m645.pdb RMSD 3.33 A
m646.pdb RMSD 5.30 A
m647.pdb RMSD 3.16 A
m648.pdb RMSD 7.98 A
m649.pdb RMSD 3.85 A


Conclusion of the validation. Across 18 poses from two runs (exhaustiveness 16 and 64), no pose came within 2.0 Å of the crystal pose in the mutant structure; the best was 2.45 Å. The correct pose was never sampled, increasing search effort changed nothing, and receptor preparation was comparable between structures. I therefore treat this as a real limitation of the protocol rather than a technical artefact.
Taken together with the wild-type result of 1.95 Å, the fairer interpretation is that this ligand is near the limit of what rigid-receptor docking reproduces reliably in either structure WR99210 has a long flexible linker with many rotatable bonds and the wild-type pass was marginal rather than convincing.